**1.1**
La primera parte explica como la naturaleza a traves de la evolucion a conseguido una perfeccion en diferentes especies, al observarla se da cuenta uno de que tan complejos somos e imaginarnos el tiempo que tomo llegar a este punto, viendola noramlmente llegamos a puntos de satisfaccion con la eficacia, claro que hubo errores durante el tiempo pero simplemente se paso las cualidades que hicieron que sobrevivieran, asi haciendolas "perfectas", tambien se tiene que notar que esta evolucion fue influenciada por el entorno, asi consiguiendo mutar, a veces son cosas que le nieguen sobrevvir o puede ser una cualidad que sea pasada a futuras generaciones.
**1.2**
Se habla de como es este pase de genes, es decir lo que hace la evolucion posible normalmente son brincos infitesimales a menos de que se aceleren con seleccion artificial. Antes era a traves de las mejores aptitudes que hacian la supervivencia del ser humano posible, tambien esta herencia no es perfecta, se puede heredar como cosas buenas o malas, dependiendo de la dominancia de cada gen.

In [ ]:
import random

class Individuo:
    def __init__(self, nombre: str, genes: dict, generacion: int = 0):
        self.nombre     = nombre
        self.genes      = genes   # genotipo
        self.aptitud    = 0.0
        self.generacion = generacion

    def calcular_aptitud(self, entorno: dict) -> float:
        coincidencias = sum(
            1 for gen, alelo in self.genes.items()
            if entorno.get(gen) == alelo
        )
        self.aptitud = coincidencias / len(self.genes)
        return self.aptitud

    def mutar(self, alelos_posibles: dict, probabilidad: float = 0.1) -> None:
        for gen in self.genes:
            if random.random() < probabilidad:
                opciones = [a for a in alelos_posibles[gen] if a != self.genes[gen]]
                if opciones:
                    self.genes[gen] = random.choice(opciones)

    def reproducirse(self, otro: "Individuo", nombre_hijo: str) -> "Individuo":
        genes_hijo = {
            gen: random.choice([self.genes[gen], otro.genes[gen]])
            for gen in self.genes
        }
        return Individuo(
            nombre=nombre_hijo,
            genes=genes_hijo,
            generacion=max(self.generacion, otro.generacion) + 1
        )

    def __repr__(self) -> str:
        return (
            f"{self.nombre} (gen {self.generacion}) "
            f"aptitud={self.aptitud:.2f} | genes={self.genes}"
        )

In [ ]:
class Poblacion:
    """
    Grupo de individuos que evolucionan juntos.

    Atributos
    ---------
    individuos       : list[Individuo] — Personas del grupo.
    entorno          : dict            — Rasgos favorecidos por el ambiente.
    alelos_posibles  : dict            — Variantes posibles de cada gen.
    generacion       : int             — Generación actual.
    """

    def __init__(self, individuos: list, entorno: dict, alelos_posibles: dict):
        self.individuos      = individuos
        self.entorno         = entorno
        self.alelos_posibles = alelos_posibles
        self.generacion      = 0

    # ------------------------------------------------------------------
    # Evaluación
    # ------------------------------------------------------------------
    def evaluar(self) -> None:
        """Calcula la aptitud de todos los individuos según el entorno."""
        for ind in self.individuos:
            ind.calcular_aptitud(self.entorno)

    def mejor(self) -> Individuo:
        """Retorna al individuo más adaptado."""
        return max(self.individuos, key=lambda i: i.aptitud)

    # ------------------------------------------------------------------
    # Evolución: una nueva generación
    # ------------------------------------------------------------------
    def evolucionar(self, probabilidad_mutacion: float = 0.1) -> None:
        """
        Genera la siguiente generación:
        1. Evalúa quién está mejor adaptado.
        2. Los dos más aptos se reproducen.
        3. Los hijos pueden mutar.
        4. Reemplaza la población.
        """
        self.evaluar()
        ordenados = sorted(self.individuos, key=lambda i: i.aptitud, reverse=True)

        hijos = []
        for i in range(0, len(ordenados) - 1, 2):
            padre = ordenados[i]
            madre = ordenados[i + 1]
            hijo  = padre.reproducirse(madre, f"h{self.generacion+1}_{i//2+1}")
            hijo.mutar(self.alelos_posibles, probabilidad_mutacion)
            hijos.append(hijo)

        self.individuos  = ordenados[:2] + hijos   # mejores padres + hijos
        self.generacion += 1

    # ------------------------------------------------------------------
    # Cambio de entorno (nueva presión evolutiva)
    # ------------------------------------------------------------------
    def cambiar_entorno(self, nuevo_entorno: dict) -> None:
        """Cambia el entorno — simula migración o cambio climático."""
        self.entorno = nuevo_entorno

    def __repr__(self) -> str:
        apt_media = sum(i.aptitud for i in self.individuos) / len(self.individuos)
        return (
            f"Generacion {self.generacion} | "
            f"{len(self.individuos)} individuos | "
            f"aptitud media: {apt_media:.2f}"
        )